**Data Sourcing** is the foundational starting point of any ETL/ELT pipeline. It refers to identifying, connecting to, and extracting data from the various places where it is generated.

In data engineering, the golden rule applies: **"Garbage in, garbage out."** If your data sourcing layer is unstable, poorly structured, or missing context, no amount of transformation downstream can save your pipeline.

---

### **1. The Main Categories of Data Sources**

Data can originate from dozens of different systems. In an enterprise or data science project, you typically deal with five primary categories:

* **Transactional Databases (OLTP):** Operational databases where applications store live user data (e.g., PostgreSQL, MySQL, MongoDB, Oracle).
* **SaaS & Third-Party APIs:** Cloud services that run your business operations (e.g., Stripe for payments, Salesforce for CRM, GitHub for code repositories, Google Analytics for web traffic). Accessed primarily via **REST APIs** or **GraphQL**.
* **Files & Object Storage:** Flat files dropped into cloud buckets (AWS S3, Google Cloud Storage), SFTP servers, or local directories. Formats range from structured (CSV, Parquet, Avro) to semi-structured (JSON, XML).
* **Streaming & Real-Time Event Sources:** Continuous data feeds like user clickstream logs, IoT device telemetry, or application logs flowing through event brokers like **Apache Kafka** or AWS Kinesis.
* **External / Open Data & Web Scraping:** Public datasets, financial market feeds, or unstructured web pages extracted via scraping tools (BeautifulSoup, Scrapy, Selenium).

---

### **2. How We Connect to and Extract Data**

Different sources require different extraction strategies based on how the data is exposed:

| Extraction Method | How It Works | Best Used For |
| --- | --- | --- |
| **API Polling (Pull)** | Your script sends an HTTP GET request to an endpoint on a schedule (e.g., every hour). | SaaS tools, weather data, public APIs. |
| **Webhooks (Push)** | The source system actively *pushes* data to your endpoint the exact moment an event happens (e.g., a payment succeeds on Stripe). | Real-time notifications, event-driven workflows. |
| **Database Querying** | Running `SELECT` statements directly against a read-replica of an operational database. | Extracting relational tables on a batch schedule. |
| **Change Data Capture (CDC)** | Monitoring database transaction logs (like PostgreSQL WAL or MySQL binlogs) to capture every `INSERT`, `UPDATE`, and `DELETE` in real time. | Low-latency synchronization without overloading the production database. |

---

### **3. Key Engineering Challenges in Data Sourcing**

When building data sourcing connectors, you will quickly encounter real-world friction points:

* **Rate Limiting & Pagination:** APIs rarely hand over millions of records at once. You must write pagination logic (offset/limit, cursor-based, or token-based) and handle HTTP `429 Too Many Requests` errors with exponential backoff.
* **Schema Drift:** The external API or source database suddenly changes—a column is renamed, a data type changes from integer to string, or a field is deleted. If your pipeline isn't resilient, it will crash.
* **Authentication & Security:** Managing API keys, OAuth 2.0 refresh tokens, SSH keys, or SSL certificates securely using secret managers rather than hardcoding credentials into code.
* **Network Reliability:** Source servers go down, networks drop packets, and timeouts happen. Robust data sourcing requires built-in retry mechanisms and fault tolerance.

In [16]:
import certifi
import urllib3
from urllib3 import request
import json
import sqlite3
import pandas as pd
from io import StringIO

### **Sourcing CVS Data**

In [2]:
df = pd.read_csv('./Data/h9gi-nx95.csv')

df.head()

,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,off_street_name,cross_street_name,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code1,vehicle_type_code2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,2022-07-20T00:00:00.000,1:25,NaN,NaN,40.835808,-73.89083,"\n, \n(40.835808, -73.89083)",BOSTON ROAD,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4547589,Sedan,Sedan,NaN,NaN,NaN
1,2022-07-21T00:00:00.000,5:20,NaN,NaN,NaN,NaN,NaN,FDR DRIVE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4548075,Sedan,Sedan,NaN,NaN,NaN
2,2021-04-14T00:00:00.000,5:32,NaN,NaN,NaN,NaN,NaN,BRONX WHITESTONE BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4407480,Sedan,Sedan,NaN,NaN,NaN
3,2021-04-13T00:00:00.000,21:35,BROOKLYN,11217.0,40.683580,-73.97617,"(40.68358, -73.97617)",NaN,NaN,620 ATLANTIC AVENUE,...,NaN,NaN,NaN,NaN,4407147,Sedan,NaN,NaN,NaN,NaN
4,2021-04-15T00:00:00.000,16:15,NaN,NaN,NaN,NaN,NaN,HUTCHINSON RIVER PARKWAY,NaN,NaN,...,NaN,NaN,NaN,NaN,4407665,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN


### **Sourcing Parquet Data**

In [3]:
df_parquet = pd.read_parquet('./Data/yellow_tripdata_2022-01.parquet')

df_parquet.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,1.0,N,142,236,1,14.5,3.0,0.5,3.65,0.0,0.3,21.95,2.5,0.0
1,1,2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,1.0,N,236,42,1,8.0,0.5,0.5,4.00,0.0,0.3,13.30,0.0,0.0
2,2,2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,1.0,N,166,166,1,7.5,0.5,0.5,1.76,0.0,0.3,10.56,0.0,0.0
3,2,2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,1.0,N,114,68,2,8.0,0.5,0.5,0.00,0.0,0.3,11.80,2.5,0.0
4,2,2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,1.0,N,68,163,1,23.5,0.5,0.5,3.00,0.0,0.3,30.30,2.5,0.0


### **Sourcing API Data**

In [4]:
# 1. Define the target API endpoint URL
# This queries the NYC Open Data portal for a JSON dataset 
# and restricts the return payload to a maximum of 500 records via '?$limit=500'.
url = 'https://data.cityofnewyork.us/resource/h9gi-nx95.json?$limit=500'

# 2. Setup the urllib3 PoolManager with secure SSL certificate verification
# 'certifi.where()' points to a reliable, bundled set of root certificates, 
# ensuring secure HTTPS communication without SSL handshake errors.
http = urllib3.PoolManager(cert_reqs='CERT_REQUIRED', ca_certs=certifi.where())

try:
    # 3. Make the HTTP GET request with an explicit timeout (10.0 seconds)
    # This prevents the 'ReadTimeoutError' you encountered previously by giving 
    # the server up to 10 seconds to respond before abandoning the request.
    response = http.request('GET', url, timeout=10.0)

    # Print the HTTP response status code to the console
    print(f'Status code: {response.status}')

    # 4. Conditional check for a successful response
    # A status code of 200 means the server successfully processed the request.
    if response.status == 200:
        # Decode the raw binary byte response into a UTF-8 text string, 
        # then parse that JSON string into native Python dictionaries/lists.
        data = json.loads(response.data.decode('utf-8'))
        
        # Flatten the semi-structured JSON data into a clean, 2D tabular Pandas DataFrame.
        df_api = pd.json_normalize(data)
        
        # Display the first 5 rows of the DataFrame. 
        # Note: 'display()' is ideal for Jupyter Notebooks; use 'print(df_api.head())' in standard scripts.
        display(df_api.head()) 
    else:
        # If the server responds with a non-200 code (e.g., 404 or 500), print the error code.
        print(f'Error: Server returned status code {response.status}')

# 5. Exception handling blocks to catch network errors gracefully
except urllib3.exceptions.MaxRetryError as e:
    # Specifically catches timeout or connection exhaustion errors and prints a helpful message 
    # instead of crashing with a massive traceback.
    print(f"Connection failed: The server took too long to respond or is unreachable. Details: {e}")
except Exception as e:
    # Catch-all safety net for any other unexpected Python errors.
    print(f"An unexpected error occurred: {e}")

Status code: 200


,crash_date,crash_time,borough,zip_code,latitude,longitude,on_street_name,cross_street_name,number_of_persons_injured,number_of_persons_killed,...,location.human_address,contributing_factor_vehicle_2,vehicle_type_code2,off_street_name,contributing_factor_vehicle_3,vehicle_type_code_3,contributing_factor_vehicle_4,vehicle_type_code_4,contributing_factor_vehicle_5,vehicle_type_code_5
0,2026-10-31T00:00:00.000,18:13,BROOKLYN,11216,40.68039,-73.949554,FULTON ST,NOSTRAND AVE,1,0,...,"{""address"": """", ""city"": """", ""state"": """", ""zip""...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-10-31T00:00:00.000,11:30,QUEENS,11412,40.69509,-73.74573,118 AVE,FRANCIS LEWIS BLVD,0,0,...,"{""address"": """", ""city"": """", ""state"": """", ""zip""...",Does Not Apply,4 dr sedan,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-10-31T00:00:00.000,12:30,QUEENS,NaN,40.76925,-73.90911,41 ST,ASTORIA BLVD N,0,0,...,"{""address"": """", ""city"": """", ""state"": """", ""zip""...",Does Not Apply,4 dr sedan,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-10-31T00:00:00.000,21:00,BROOKLYN,NaN,40.661743,-73.94783,E NEW YORK AVE,NEW YORK AVE,1,0,...,"{""address"": """", ""city"": """", ""state"": """", ""zip""...",Driver Inattention/Distraction,Moped,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-10-31T00:00:00.000,12:00,BROOKLYN,NaN,40.672585,-73.96209,NaN,NaN,0,0,...,"{""address"": """", ""city"": """", ""state"": """", ""zip""...",Does Not Apply,NaN,425 LINCOLN PL,NaN,NaN,NaN,NaN,NaN,NaN


In `urllib3`, the **`PoolManager`** is the master controller responsible for **handling, optimizing, and reusing HTTP connections** (a process known as *connection pooling*).

Instead of opening a brand-new connection to a server every single time you make a request, `PoolManager` manages a pool of active connections behind the scenes.

---

#### **Key Roles of `PoolManager`**

##### **1. Connection Reuse (Performance & Speed)**

Every time your code makes an HTTP or HTTPS request from scratch, your computer has to:

* Look up the server's IP address (DNS lookup).
* Establish a TCP handshake.
* Complete an SSL/TLS handshake (for secure `https://` sites).

These steps take precious milliseconds. A `PoolManager` keeps connections open (**HTTP Keep-Alive**). If you make multiple requests to the same website, it instantly reuses the existing open connection, making your code significantly faster.

##### **2. Centralized Configuration**

When you initialize a `PoolManager`, you can pass global settings that apply to every request made through it. In your code, you used it to handle secure SSL verification:

```python
http = urllib3.PoolManager(cert_reqs='CERT_REQUIRED', ca_certs=certifi.where())

```

By setting this at the manager level, every request made via `http.request(...)` automatically inherits those security certificates without you needing to rewrite them each time.

##### **3. Multi-Host Management**

Even though you are only querying NYC Open Data right now, a single `PoolManager` can intelligently manage connections to **multiple different websites simultaneously**. It automatically creates, tracks, and isolates separate connection pools for different domains (e.g., one pool for `data.cityofnewyork.us` and a completely separate pool for `api.github.com`).

##### **4. Automatic Lifecycle Management**

`PoolManager` handles the messy parts of network programming for you:

* It closes idle connections when they are no longer needed.
* It retries failed or dropped connections safely.
* It ensures thread safety if you are running requests across multiple background threads.

---

#### **Real-World Analogy**

Think of `PoolManager` like a **taxi dispatch office at an airport terminal**:

* Without a dispatch office (making raw, unpooled requests), every single passenger has to walk out, buy a brand-new car, build a road, drive to their destination, and throw the car away.
* With the `PoolManager` (dispatch office), a fleet of cars is kept ready and waiting. When a passenger (your request) arrives, an active car is assigned immediately. When they return, the car goes back to the lot, ready to serve the next passenger instantly.

### **Sourcing DB Data**

In [5]:
with sqlite3.connect('./Data/movies.sqlite') as conn:
    df_db = pd.read_sql("SELECT * FROM movies", conn)

df_db.head()

,id,original_title,budget,popularity,release_date,revenue,title,vote_average,vote_count,overview,tagline,uid,director_id
0,43597,Avatar,237000000,150,2009-12-10,2787965087,Avatar,7.2,11800,"In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,19995,4762
1,43598,Pirates of the Caribbean: At World's End,300000000,139,2007-05-19,961000000,Pirates of the Caribbean: At World's End,6.9,4500,"Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",285,4763
2,43599,Spectre,245000000,107,2015-10-26,880674609,Spectre,6.3,4466,A cryptic message from Bond’s past sends him o...,A Plan No One Escapes,206647,4764
3,43600,The Dark Knight Rises,250000000,112,2012-07-16,1084939099,The Dark Knight Rises,7.6,9106,Following the death of District Attorney Harve...,The Legend Ends,49026,4765
4,43601,John Carter,260000000,43,2012-03-07,284139100,John Carter,6.1,2124,"John Carter is a war-weary, former military ca...","Lost in our world, found in another.",49529,4766


### **Sourcing Webpages Data**

In [17]:
# 1. Define the Wikipedia URL
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'

# 2. Add a standard User-Agent header to mimic a real web browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# 3. Fetch the webpage content manually using requests
response = urllib3.request('GET', url=url, headers=headers)

# 4. Check if the request was successful (status code 200)
if response.status == 200:
    # Pass the downloaded HTML text into pandas read_html
    df_html = pd.read_html(StringIO(response.data.decode('utf-8')), match='by country')
    
    print(f'Successfully loaded! Found {len(df_html)} table(s).')
    
    # Display the first table
    display(df_html[0])
else:
    print(f'Error: Status code {response.status_code}')

Successfully loaded! Found 4 table(s).


,Country/Territory,IMF (2026)[1],World Bank (2025)[6],United Nations (2024)[7]
0,World,128003776,118350166,111565144
1,United States,32642373,30769700,29298000
2,China[n 1],21546403,19498039,18743802
3,Germany,5471299,5050923,4659929
4,Japan,4341986,4435163,4026211
...,...,...,...,...
217,Palau,377,345,310
218,Marshall Islands,342,308,281
219,Nauru,196,176,187
220,Montserrat,—N/a,—N/a,81


In [19]:
display(df_html[1])

,vteLists of countries by financial rankings,vteLists of countries by financial rankings.1
0,Trade,Account balance Exports by product and service...
1,Investment,FDI received FDI abroad
2,Funds,Forex reserves Gold reserves Sovereign wealth ...
3,Budget and debt,Government budget PPP % of GDP per capita Cred...
4,Income and taxes,Wage average Disposable minimum Wage growth GN...
5,Currency,Central bank interest rate Commercial bank pri...
6,Other,Financial Development Index Average annual lab...
7,Lists of countries by GDP rankings List of int...,Lists of countries by GDP rankings List of int...


In [20]:
display(df_html[2])

,vteLists of countries by GDP rankings,vteLists of countries by GDP rankings.1
0,Nominal,GDP (nominal) Informal Past and projected Sect...
1,Purchasing power parity (PPP),GDP (PPP) Past Past and projected Ten largest ...
2,Per capita,nominal GDP Past and projected GDP (PPP) by re...
3,Economic growth,real GDP growth rate African countries Asian s...
4,Productivity,Labour productivity
5,Gross national income (GNI),PPP per capita Nominal per capita Real income ...
6,Countries by region,Africa PPP nominal Arab League nominal and per...
7,Subnational divisions,Albania Argentina Armenia ASEAN Australia citi...
8,Lists of countries by financial rankings List ...,Lists of countries by financial rankings List ...


In [21]:
display(df_html[3])

,vteEconomic classification of countries,vteEconomic classification of countries.1
0,Developed country Advanced economy Developing ...,Developed country Advanced economy Developing ...
1,Three/Four-World Model,First World Second World Third World Fourth World
2,Gross domestic product (GDP),Nominal By country past and projected per capi...
3,Nominal,By country past and projected per capita per c...
4,Purchasing power parity (PPP),By country future estimates per capita per cap...
5,Gross national income and Income,Disposable income GNI (nominal) per capita GNI...
6,Wages and salary,Average wage Europe Employee compensation (per...
7,Wealth,Wealth inequality Wealth per adult Financial a...
8,Other national accounts,Gross National Happiness Net material product ...
9,Human development,Human Development Index by country inequality-...
